# CrackProof: QLoRA Fine-Tuning on Google Colab (Free T4 GPU)

**Objective**: Fine-tune an open-source LLM (`Qwen/Qwen2.5-3B-Instruct`) using **4-bit QLoRA (PEFT)** to act as the specialized **CrackProof Technical Interview Evaluator**.

### Why Fine-Tune for Technical Interviews?
1. **SOLO Taxonomy Rubric**: General LLMs give generic chat advice. Fine-tuning forces structured evaluation on 4 depth dimensions: `FUNDAMENTAL`, `REASONING`, `APPLICATION`, `EDGE_CASE`.
2. **Strict JSON Schema**: Guarantees deterministic JSON output with `verdict`, `correctness_score`, `depth_score`, and `missing_core_concepts`.
3. **RAG Grounding**: Model learns to cite numbered reference passages `[1]`, `[2]` rather than hallucinating facts.

## Step 1: Check NVIDIA GPU & Install Dependencies
Ensure your Colab Runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Verify NVIDIA T4 GPU is active
!nvidia-smi

# 2. Install ML Dependencies
!pip install -q -U peft bitsandbytes datasets trl accelerate
print('\n[OK] ML dependencies installed successfully!')


## Step 2: Unpack and Verify Frozen Grounded Dataset
Unpacks the verified, frozen canonical dataset across 6 core subjects (**Java, OOP, DBMS, OS, CN, DSA**):
- **Train (108 samples)**: 36 canonical questions (6 per subject)
- **Val (18 samples)**: 6 canonical questions (1 per subject) for evaluation & checkpoint selection
- **Test (18 samples)**: 6 canonical questions (1 per subject) — strictly frozen for benchmark comparison


In [ ]:
"""
CrackProof Phase A Dataset Loader & Integrity Verification.

Unpacks and verifies the frozen canonical dataset across 6 core subjects:
- Java, OOP, DBMS, OS, CN, and DSA.

Exact Partitioning:
- Train: 108 samples (36 unique questions, 6 per subject)
- Val:   18 samples (6 unique questions, 1 per subject)
- Test:  18 samples (6 unique questions, 1 per subject) - FROZEN FOR BENCHMARK
"""
import os
import json
import gzip
import base64
import hashlib
from typing import Literal, List
from pydantic import BaseModel, Field

# Pydantic schema for evaluation (self-contained for Colab)
class DepthEvidence(BaseModel):
    evidence_type: Literal["FUNDAMENTAL", "REASONING", "APPLICATION", "EDGE_CASE"]
    status: Literal["DEMONSTRATED", "PARTIALLY_DEMONSTRATED", "NOT_DEMONSTRATED"]
    evidence_from_answer: str

class Citation(BaseModel):
    claim: str
    source_number: int

class AnswerEvaluation(BaseModel):
    verdict: Literal["CORRECT", "PARTIALLY_CORRECT", "INCORRECT"]
    correctness_score: int = Field(ge=0, le=10)
    depth_score: int = Field(ge=0, le=10)
    correct_points: List[str]
    missing_core_concepts: List[str]
    deeper_concepts_to_probe: List[str]
    misconceptions: List[str]
    evidence: List[DepthEvidence]
    citations: List[Citation]
    reasoning: str

os.makedirs("dataset", exist_ok=True)

# Embedded exact gold-standard datasets (gzipped base64)
TRAIN_B64 = "..."
VAL_B64 = "H4sIAAAAAAAC/+1dbXPbOJL+fr8ClQ8Xu0a2J3u7d3XOhytHlh3POnbWcia1N55KQSQkIaIILkFK0UzNf7/uxgtBSrJly06cOX6YSSKSeOl+uvsB0AB+f/GvUuhCqvSTjF8cshc/Hf189OnH/3rRYS90OfgsooJ+5TOOP02F1nwkNPz2y+8vcpUIfKoXuhBTfB6ptBApffJPVTKeC8aZLnIZFawQ0TiVEU+YhJfymRRzke/fpPBizgquJ0xqVigmZjwpeSFYMRYs4mksY/jXS82ue923F2fdo3P24eK4d9W/Pro4Prs4hSJ67pMB1yJmKj28SV/ts2tfY6TyHLqSQvPZzo97r37cvUn/ss+6Ko1EVpTwRgx/jv2z/9hn/cvzS3bNv6hUTRfsmB4fy6lINQhLH7ITaMTRu97F9dF5h131jvqXF9CYDjt6//4cGnl9dnnRYb3j096n7lG/d5P+1VenXXOSBfxtJnIRd9hUai3TET6CXrsX/z2QGrxhf8cG3KR/22enuSpBPunokHUldD8XQygN3mEZN4piv7z6tcN++cuvbD4WKctyNZOxiEFmx4pdXF6zTKQ8kb8JNsr5dMpzaHM6SqQes2FSQlGLDuNRBDrtMJWztxKq23NvRCoWe3oui2gMbYAyj5I5X2imyiIrCwZKkTH7CQSDHRqqfIodBA2jYrs5jybvc6WG7CjVgASrQ+gb09FYTPn+iz86rAJZqUXegNjZxXXv6uez3kf2jw+9Pkoc9P5WzVmsoOeIWXbK8wHIAUSfJCBxLH2u8gl0Ko1BJLwg0EF7Yjl0shuIYi5AWPgzgBPafCpSkZu24XeXSRz+JFM2FjxjUzFV+eJ/btKbtIvQPD667rGji/7H3hW0yzSny3hZqCl8CCoF/QMMEi6n2n7MQBxlmgsejfkgEUyRBWpWEjbgaxAYWBLIDgXOPqSxyKmdI98cQMp4kSn4UUsNuFK68MXEUrAF9qjDtDKNhu7rLJHFYdVVttOL4f8/sH4JVjoDteuMR0LvIoYAnamY+xLRwiNobwFmh6KJEsFT+DuAAhDEoe53MoUSTru6E8rOFpWodLSXyBkabVAiwHSqqMgheAqmqR1kHQuoAEpy1QwW7B3/bMrfR8Ff9U56V72Lbo+9A+lfnR2dg+jBBtaioa6OKU/JagJ9Yh1gMmkhhwtsAnYiFoko8B936krvszfWJ5GePgo+CbADynobKAvfqGkFXaU6XEah0RA2paGjNSqCzqmoUpLpPvxrJjkb5gJiQBpoancVyneuRVqCqzIV2XpcHZWGpmVSyCwRrhL0VTWddhrVJ+iTqzagNg9OyiShhjR8ADg1qQsO1l93BL/fpIzdvABXGkOkuYGfbl50L6+uet3rmxcd8zAIAZ80ell87dWP9im5/+r3/65/9ClToAiNT37BB/gIFIx2CCJfhsAQ+uoRYFtA36y20sO6lY75TDA9VnnBEjkUIO80LKSy0zHXbC0QwN96jQZfO/vTBXRVrzLAtcZXjOGLsUrimxdY2q9WRjZyfULZfXKRi0T1qxeuyETun30q1CeoZCBq8uwXKtsDeex9VDm0MePg8MNuv4Ngk6fkQZORysG/TEFwp6867H+xg40WhZEybIrA8AePgqp/N3+ETz8Vi0wYGAVh3jcGXwUYFqU27xz33l1e9K+vwOMc117y5SEgPnGKcxad4MBysAtwPGViDAxcP9l+5co1KdbCS4I7WOybjjL2R2eTxnta8qhNV2lqzPeWwINhfkrom0Fssy7amXkU9CPUWyQLXqlsqYMUKU0bNneI+/v7T+nvsHhnMLt1KQOtBfGl5XRgJPdqRY9Bt1qlUI7p1jUYvSpHY2oP0K4yR1IbCx3lMnMgoUhWE/vIhrXK5ZJ8/3jxx69//NvvLcNvGX7L8L8Cwz+tmyEI0zJIzfmQTcDZcwjZkn3mA1PcYAwx4DM3P+/bllSWDZ+zlC8qrzMA5iPwXaStOTWy/nYGDiOt3s/F2L7fcuOWGwfc+P3R1TWo//yfnzZiyf+5miT/bSOSTEO1GmqWuOwIuSzwSl3XjZVz9RT+555uSkFdResxAUQS7MmjKGScVudIYdwoE7wIxNI0sjTFv3sVkLQm/Q9buhEVfk9wkGQWlnKznQIhBp3c/bZct4LOQ6gjh1CMdjKWo/FeImYicZGQDSCeoB7hKbEH4pMIGiAXZYSfhfZpHB6IHCKn/rps8gntf3sG+bYSbImTQ+gRkD5AFUJTjEc5JxCZdUA8DDXKBRBJetnaRkPeGYw3hW6pZUstW2r5LSaPqWGpKswEkSdnK4Z/rwGR4AIUhBoYfZe6QPJWEotDLmc4mkCdALkQLJETgQ3p/vAD/iYSLfx8NCBsmKh5O7vaMsiQQZ5dbDa/upo5vlrPHDendIQ5nAK9zRICitb7AjoHUuCQFwurNzJDY1lgMFNZFMRBCNQPYG8//fzOVTEFH5fcydfch29EIgWpmbqGSpM4R2tsl0z1wFhuvcgn4Xng3h/C8E4gpHCIcwU2mIMjGSsVB8wCe/ZS1+SzLXv7ar5na3L2JuFoXtAaGEEY6DPqCOMDVTon7YVDLUZZOjDeRr0uL98vMy/4sSVeLfH6DojXx9vZVL8g//5GmsHMTo/noAb7TxNQjxcpn4bvnCO0/CtgQmAN6/iVLX/gvhVU/sB9rHAqHgINmKuaZjIRewVgykZpbPJUFGPwc2gHX0RUQsXQlxg8NUoP16lDljLjuUS/8lIHqkfvvM/OCsYzCFOCbEpTqzqgfzmDzhjeOZQp+QisENfUgZQIiOfwJ4IyURyA4h7ve6n4niVkcMsdy8t0dZ+IMEpsqlbJrNkXHpEduq+NpwRhI4+AXhm2xoGK5EXpmw1+A93qzoz+3O1ApTIaM8A0/FO7l7A7uYwNRm/hnndgY73uTA/srzkJRS/1PJbW5q0FDBZg22BBjMdxjo6pJo66Pl87x30AzBr5J4gxUK9Xq9NzU7VpfKtO1yB9pU5tX7+9Ls2Spq0iU8kCYl0Ggpk+46X+hndYgygvyzoIAgZ6dItpk+7X6Twoo2nQy9r27bD6bOgxKOuDhsbcqk5qapllmItQ6fBJ1v/fw0ATgg1BbUUEMHbIPpfTjIRkGgg+1kmb2uW79tPZNfQ4kSnJaAhMEwgBIS0yVqAh8urvM20AnJFPHBBxHYQzXeHAImTFNOm3Sx440lpFktqt61ZlMsbQCg7MkwNrG4aXN2BPb1sMhIFiu6HEg0IJzgnf1yXjNxv5063HHCcJn+MkQIfanIsxUuGZAJ6SJTDAcEkFDbDoqbaMqR1ztGOOdszxFcYc1qUwQ4wEG6tCUroALvU33Z8nUNV7+6xRIDy2LnYiFnOVxyzhIzB50HhsMxEw/2BYpnZ5R7Q8u+XZT5828Nd7E+4xKFOkIbbISm5hxcEXVgb3TRhoNIEnWq3E1K28nWgKzuzVDfGWhueCagCkWHKDabxWq7go4QcE956W/rkOp1jqjIM7fsDsdIO4+XlqxHbD40A/BDR+YWv9CvPWW+UnnOLmCknBJ5ejsUlKwIhUaq9UnLSWQynyrdMOHsdPbc0Qu55i2Mlwiam9JIWOy8oo+ESkSEKw9bjOhSOowk+uD5BceNDTguMAx2bweu42g7w0779sCWVLKFtC+RUIpcTFcoDkkJb5qekxe3v97nzlgNq9TbqKTIq/H2i/X4BlpwDxCNxA3NLEliY+89wAC8KZXh45ISRI3+vmzayMSCmly9EBs3sA54KvXHEyndlcg40ZV5eIhxnKURn1qbIySfZAJdGEzcWAAtsoX1HBc8oO6ILIMX8BuQZoN4F4mxbU/FjMAIo8UaPFdzOJt/12H1UQfaqyAabgj2ykQqnwHMIbODjMfr2NNx2/eddfJk74a8ucWub0/TMnnrKzi4veFfvp8gwUdN47uWaXH679DxjETj6cnwc/otPu/+OcgaXkEODXcaajsGRQBVhaisNvZBhIINRcV0viOS3rc7BSIzwc5mI9AwUwo4ALdOCo2TxfKkZ7KpD2B2DxiRgWNmJnSamrguuvmbEovQccZsi4+RBeovy3VJkPO8HLEcSuqc0NHMokcaOwC5ASNbIprtWNDHrWAXjKnKJRrZVGOpkCS8B+oCqAo4ic2CNVR7MAEuXHNOCYQI2zAk6SJNQBZqffzizPjEYhbrjsTp6MxCDnh+wWNYLNqpyYJLeZs779CF6hvQY/K8rVMN2tczF8hCn0QHLAY92pZFvlCj0DVnHLtVFGJcjmB02NOxUz8QVoGOgCBevaT/mfKrPLgUj36fM9AyyLhDt1bpvgyGkN6ygiq0HbC8QUYcF6QHrNbqdZqfVnvLx/F3K8muqmHsziERiWROqMVFfiKlMjqLhhpl5sQaEPsNA7tdJQypMs5HevLvt90+SdLs8LoSWnIBaXUbEbrtGjUakMp5p+MzHEsF3wacYYNW0eF/nIGOD3uVyPSDKbxkYlxF8cOZJJ2WlTzO80O21IPxmMY1EGAwGuSoIp8ygH10pKx81gVhLbcuQHO0vkyQ90ffjpo3mgrbn3MU0kdPwu+47N3fcDFE1zmo1lcow/FAs0cB0YBES6peUtLW9p+WPT8rM0BUsnS5vSnOAUuhLjTnnut8ufo2dxrwRMesJxw33O6+/TMnpAq/C11eXS9hhXruYDNinB/8Ql7h5Buxiras9+y1dbvvrtlskDK3GisZD2mkJsh0TV28wSUbX5lIjsxliUdGsKut8aeo3RoJ4r+mu1HNJdb3Y4ztTLYDPJ70mivRJRTzUV7sii2iVZWSwtQeM2/+bm+Y34LPViDLUndgwALtmKXW8+l4vci8SA3axKYC99M3XQzpdsJ/Kzv9TBiklX/Hr3ua+nH7kjjACSBqyoRA9C3akvsBMELkk2NDZodPy9GUJsy32/qlfZmqeuEBvwq6FMZbX5k6jq/WRZGUcWSLXlsS2PbXns0/BYaY5kzHIJEl3g1DG4v8qm7WP4T8hRah9TxT4uSssQw3X6zO5cZZi5k5v4jYBHkeKvX+xJKrh0X2YszuVMtKy1Za3Pc0c/zfmhoAP04SEWvFA5cduB2dBD058eOh4xoDHwRUDpUkxkw/mabLzQ5CgN/h+SN3m1ZAj3pH5XwgQ7bLiZ1cRNVZmI5NBkSlJyiuG33G6P8UzRfED6tiZcFnjIlGxuXnpOy/y0ro2grR9BBPoI9Iqec/FIs5lP47JwwvKy4voMuE8sZzRtvnXKpZlmZEMuEzzZChBRyaqaZTTYRxncmjW5gpxdttSspWZ/BmrG3oEFfjHOkfWhnmyMIkc6hLDcAxFFCBO9SKNxDjZmlpLWrvfb8kzULQlL4kuUlAgTlqhoUtsFZzyvmgOL02OZVdP+N+Vffnz1V58ngEszPDauBAuxhyNPfV1JouYiNnaOLzAJGhIo5cik1JPNIaQwIcmUhorUZGOcRbmkZGtogTngF4hGIA3sjQbeyJPl5uP+A9+Djl2v5+nCNRps1n57kOEx5NTGATWSszmXhTv7m8c4OZoj2ZhCrR1slsJUKOyAb8uUg2I4BXKjoxE4UHqvY8Rg0hZtAASZGM+J0QHtpQRTg9ht6gPrv6A9yFYU8NmYMsoxOQ/dLjAyldzJZxvYgKhgUbDzziCg5xFgsix3jUQ3h8PrKmXER/YQEzyixb74YAkcKH3MhANPeOChwXaA6OAoAYOxOLSQss9FvPtQ7b9eo3fbACzGlYpVdJ12+xtoF2cpnHqBpoDSLlYpNAQ1zWIYHRpyYJSqn3GugMENZt2sQAHbIRTQExKvUVi45v0QrQWfd5ftjYSubxN5ZT8N66kdr+9clXFF5vQThCCOdJreZ8lzPs2JALnEc/4XYMKgY+0O0At+Bk8kTTY0fF0oGIRZrky9qfXvjTUoLzc81JX6XEWd7y/BwE2wEjVJi5xrHJxav3VQQZSEsgJsz+JMADyYjdOkwIpwsuRbLhyutx012CCALP/bentswVZ+4cHDEDxNopkCgUfzJjizYThZlJcwIE3CVGSk39UpyOi5bed9y9rRSjtaaUcr249WTGliQr4Jd/rbCwQU/uYOAIC/jqUzwYkynrK6biD0LfCq86dhYf5UUccYwlKY5hNX0rHCKVKyBF/QBHgcz//lC+zA04mkxAnNp2KjNImWoP+/Iehf7eoBa4mGHRtenvqBOmq8eYbAem69ZB94si53Ur1vVsTlMjoPLdzdzmmHIzyhlFY2LM5N/XYMDzZYh1ha4WiPzv1dAbkHTIBfN4I7erOp49f3PjyAV5MiWk4zPMvKjTQquS/ZRTDx8sxTHj6YaVw89TaYGCKo1CJGddsBvjYMjs2tOl29/6fhultz1kC+Ec94hAPBSk6dpmDBDHBlNBTrQWUWAY1tKWtLWVvK+liU1Z0mAKX4owfwihw63tYmNjT40Zjn8Zyuk0X/YzMCpnh4fz5Q8KwlkS2JfEa5Cm+Um2ZsgBkhfNk/8EKJFk1kUg6QLORMPCQfoRsUvK6gzY5xqkPLp0oENke2+F0cJxBTj+29pJTg0xD6I+QYrHMw61kVPtnEhrc/5ZMPcmmvIHEpJAovCCZNp4qSitEcY2W8W2CzqwS1hgh1L5aJUPeiJUItEfr+idAJznd1cREF4hS6dPj7yKDf/wx05rr73s6Tu6uisLx+Yk7T+QhigN+D4KzyyTqqRDVGtmhcwKLLfsy5RZGQuEPebPEVlD0F/wZVJVMzLYCzbhqMF3PdckB0+BWPZ7j/Hk/3lri5dMZlQlmKgxK7XQ2cpGFZ0Ce8FyXGT00DrkxRrj87+TyNd02/if1QrUjxyKHZzs/Ny5jdhnyPtpI4ciK+REKYA3igqP1QuCslQJ4AuopHRbMUVARyhB+HuMLlrgMMhKN5QUsYRjS4VzbnQ6Bnr8PmBqwmrN/1McI+guw+l7SIZg8OogNaaL7H3zpu7svqo/b6Bc8LEsvR2btjowZT20vtDnDyuR85xC6yZrxHzFRq4Qj4l9NySvRontprpObp3ZwXNHdYR26WixltKQ46ToLy+DETScImm1So2YGX9wq1J+goPwvg15ujytxchaJagaCX2K+X1oAs2HQTUCCzhiFBnbi8ayfwAJxH3b9jhiimMe/urzJR339TpF7T+ZX4ymlfDgSCUaIGoDn3eyCNjaD0MqLOhggqytTeyLUeR52wqKOZgvBGUxonaOhXwuKnsGnh5kcKE4tdgzwBeoiQDq7DGnSfTLlD6Np9xnkWq31jE7IWee6Sutpk7nr/cqvuDXc07qR+f1itSeT6bDucgI2XJO00XOIy0VvZwhAwxg8hI/CvzWuOCrEU+KAD8kDhdWceDGv1/yRZG2jf13ysKMPiSqQK/3zz5irsSWUDYWqMxH5bE+ov0hg0Yc6TueAjui3Hf/YnydMIHK6HtEUpSmm2BqrVK88he+MsODQWfNAIjTG1Gz6nZcIPl7zOtvv3HiXc4dDsacIHlnxfT/xYqRx+UJDg5ICIIFrXcjUc2R2GvswcfLHkjYKrIOjCFXtrlW4HiO0AsR0gfvUBIg2ngC+S6U6U/90netAJFyvsuGKxblTEPvMpFpErn9nRbxIWzBWx/+B4sSYyGnvZRC7G9mqKz3wAeIkmosCLMyl+uTSQdujSDl2e59Dlqx3PUTPfXIxo66g2hkynIoBDcxC9a+hikeHUGhhyjULXrdhDENBnzNTeVx1NUjVPRDwS8b3vxXALXxkHt483XEJEo0mLHW9upvpdJLARPVkaxIQp8Q3N+m18jdT32lhnFdpqo4p7L6tQF2yDv+3wYqv0k7O0KGlFiNX3XgKCMBGC0iMSTquTdNte4jPnA4WaxWAvzW0J+2O6YSTX93BkW/PqU0w0AW4IgJdGtCrthFKsWJZbEqY92RWgO01+zT1gszGQzpZPt3y65dNfn08HKSpzjnkNmXS78Buzd7Rqe/rhjA0Fp4UH6byMZq9euU3auApB1tcy35b5Pk/m+wzSZk6aE08NW6MEMHS8JAy6dzjhC8o68hqweLSKXrqPeONZas98wmsWHnA7R9WQsJWUZpElJZ5ZYq5ZuexDbNOTQmUGyUMePedTPPwhFalK94Lgh8lp+eIR8mke0Q9WtHC9O9qaC17U5PBZTXB1QhN1uuXstP7RiqPT+kcth2s53PfP4d5gumcx3juRMNRkfYGelO28Oenb68xQz42Hx/gQXDhIKBujm8e93e4KzTml0ZrbNmRutrhRlCCaYk7n/kKnH63jeFC3K9RSRVNRAp4l2Rss9ugv/vasf5SiFGzn5Ozkchdgw8mLy4I+pKPpEe5jJSNzE+fQXg1ELmis8DTFAtBR0NldZToXuHEWbIbq1PvsDDlNve0LFOflzs90roUulOksVP2FUkDmMsbk42HQdJul8y9s6j47rnUQ2BXGO8Oz/CUc4PZSJjiIm9o2EHisHB0gVSBQ6GZHJ4E+XSGFx5kJvN+IzhY5I50DtKH32G0IWSpRIzPMVXS1B7CRRYRVCbsW2bEHtLrbjF0yJnZcpejVbxdHx9wiWW16chIZk1CdSHwrQQgCxEF5uHR6k9nQFZYMn0DJ7AfW23XdTXFhHRqA2bNgdXfcOHJax6im4+Dp8AAk1ZEdLuA8GrgwRJ59rjeDWyXg9bhSK3D1ej2moKzLnY+7bGc9nnYNhnxTLYRqADLH5xGCBkDEopUYakBo5xx6VQNSrYtNDK0B0JT/hgeaJpi4elc/3wb9NBFnJUh2LUoCcpaX5lY3hw5CzjqIPN8smbqz2wBz4YEY8O2Q7li8G3CGzJLgyQsZvYTXGIeAsg5pPWIILwdeSY1ywnsrm6Axc2l13ASNM9fhNRtHyl+p8QpYT5IKcyw/T4CucTStkUhFTtwgJk9hhgfuKBMjVexxQ/JBN655/pmnL+mEFrBOk0qy5GStiEGQf5IkGYLvgcG2ynFEMNMWQTVoeS99z6SYgGE+amdsGn6o7rqhEUFyj1Z4R3PwFgD2BwvXbYdcGwcougzxUUIElrSJ733K/Be0DZ5LbY4qodU3n0OsOzBsJF5qp4sijcwIR0pa6PBOGIo+xFB0O9xrh3vtcO/5DPcGtlVztHZrzlVWjAmtaNPw2wQPOQKLQe9GoF3zUUUg/Yf29JPJ8jDjEg8p3CzvpR1QtAOK4ns4PQWxZ7f3kAE1OLpLf18i8G8Mga+ZpCM6Ugc2d9+8k/c2mls/smSCVu7AH+n8dIjlhnLLiOwzTCRpQiA4APAQwY4Ap+QVAATZSfj1UWYuzjBxrG5aiEoUwEwvgfOAsGnO+F/i5hsNJk5XdjxI2tj5OfQhPfQhu99xJsvfUzEn2zPZF4TCaszYPEtlhDNAeI+2XbMgn1zJaWvqvIW5P8Jp4eZy8EYaCnpyE9xmgS12VqT4+GYGwDFh1uVyAdoSMcXDGTy4W5bbstyW5T4flmsviDFUla6ItefzysS6/ONVb8UC/WFsX6MzAfIY94RMWqbaMtWnY6rP4YgWULWzCzS/8JA1Y6iUyYSXKnucb5dE3DCatYVtlFbiP4YAbZprrNrd/4MWbVPOzN9zm1j0HaSUVMe1mPwOq45HSzl+LO+1yUzoI1zxh0kv1kc7jXKIueAyoEp7iMtvIlc+Nx5DZCPWhITt/wCTzr5E1sUAAA=="
TEST_B64 = "H4sIAAAAAAAC/+1da3MbN7L9fn8Fyh92pVuUlOc+tFX3Fi0pthJZ0opyXKk45QJJkEQ0nOEOZiRzU/nvt083MIPhQ2870t354MQmhxig+3T3AdBo/PbiX6Vxhc3SD3b4Yle9+L77Y/fDF3970VEvXNn/1QwK/lRfanw0Nc7psXH02c+/vcizxOBbN3eFmeL7QZYWJuWf/JSVSudGaeWK3A4KVZjBJLUDnShLD+WX1lyZfPt9Sg/mqtDuQlmnikyZS52UujCqmBg10OnQDulff3bq/GDv9fHhXvdIvT3ePzjrnXeP9w+PX1ETB+Enfe3MUGXp7vv0y211Xr1xkOU5DSWl7quNL7a+/GLzffrVttrL0oGZFSU9MaT/T6rvvt5WvZOjE3WuP2ZpNp2rff56305N6khYbld9R53ovjk4Pu8eddTZQbd3ckyd6aju6ekRdfL88OS4ow72Xx182Ov2Dt6n31Svc6E7yZz+dmlyM+yoqXXOpmN8RaMOD/4pkho94T9HB96n326rV3lWknzS8a7aszT83IyoNXpGzbQoSv385S8d9fNXv6iriUnVLM8u7dAMSWb7mTo+OVczk+rE/tuoca6nU51Tn9NxYt1EjZKSmpp3lB4MSKcdleXqtaXXbYUnBtnQbLkrWwwm1Adqs5tc6blTWVnMykKRUuxQfU+CwYBGWT7FAEnDUOxergcXp3mWjVQ3dYQEr0Mam3KDiZnq7Re/d1QNstKZfAFih8fnB2c/Hh68U/98e9CDxEnvr7MrNcwEYyNNohiakS6TQhGSCIr0hoGammKSDR09pIBs9Tc1tCMSnRrl2TT+VvcJvJrAO0hIoMb97/v0fboH3O13zw9U97j37uCMXnq49LbQiE4S6pAeQkv+Q2Wns8QQkgoerlMkwQnJTPVzoy/wnPloySbpL9WTJneEZ5LbhZn7zrKeyWQwJrNbD9jBaNKsUBN9SQ+k9D2eHFmTUH82qEPh7zDPWdlPSCJeMCNLaNiEpknMNPRyUGS56xB4bGKWpIEXqammN9MfNS0L3U+iN3LHeHCN1rZVN3EZwUqa4UaqcVIrSWFn3Ew1nj7JxnwkrcO0yWiydLEv29DL2cF3B2cHx3sH6g0p5+ywe0SaIfwHHVOLeTYsB+QhrsdE9Gr4o9QPK6jYXGZJyUDlsfUJyFc6H9IgpzNqqm8TW8y31Q8NTZHH6EYtVzrKkuGixDbSbFFt/wCOyKByi76s11ylqUXo3kpVl1bXH0UvS4eLCryt6jqsOzx2g/6atk6fWHSjWDD4396nSr1/QS5zSBHlPX30/sXeydnZwd75+xcd+TJy9R8cvCke+/IL/y27+frzvzd/9GGWUd8dvvkZX+Cr/ZUGfUc0+N6hvcMb7FRUsWCB0e+7i0a4XqvBzBf0FzX2plbZxORWfkYuObKAS/Iw5IqWn1n0Bu9foNVfvER9PPsASX8I8YwF+0ulCjMzefXdhyL7QOGpb5rSt3qaSf9HCVhEblyQd14CnbG9LnrfaKTvEP7ImgeTLHOL8MMgq0aWxhFH3XgABqE0HcQd/k3+F3/7oZjPjEA1ogxVz/AoVF46eWb/4M3Jce/8jDzYfuOhqj3EqA+aY6a3gIpN4InCktqXfMoCBmLHtC0DVur3zm0GUVGdxxzCazueJPSnoJ5XbmQ18khtxDqy0ajud6yvgfVRdbVWSN12Ku98gDve3t7+HI62KWGiySS6tJz2RWpfrhg9EQiXpSQ0j4zEgNHNCB6WQG8+zhKdCsciA+a+dBYiPfowXeEVWNq/v/j9l9//67d2ztDOGdo5w0PnDL6V6Yo5g05B9AmZehh1ShPmF5wNsYuLMlFOT/F0qsZ6bohP2LQDWEe/pe9XWbVnM5OssPhZS6P/Y2n0affsnFR99NOHWxHqv6zm09/ezKcj/ivgW4XLiLgt8m/vjhYm0U283ZaKXsPJAag1+r0DN4fKmN7zL0dlsp6Txz2+FTU+XIbYkvcBh+LXEAiaNnkjyw2veWkSS8QNixGR5qj7S6sBfTOAH4L9JqYwFR+lQNR82ychzjWA70OhNUX0HORz1QrNWlIkdjnS5CtgkwtAeb4E9cHE8w2ZHImE2Rgh3IIs8ZLWIJDP6+cnrqWbLd1s6eZj0s2ldSxwn9xeApPhs2KihWYwv+gD20kiiwkIuRBK3WlbOJOMOsplkCY/EIU9OCtoxaFVM22Z5X8sszw8vt0S7WpG+eV6Rnl7frcS+6Kd/ryBtaFxdpwidGXAv4/19G+WsVgD/aSSKfvDVeuft6Rw9fyMeYe3mGk2xCpeHps/OvD3e7C24TV2jzYJlvxSS1D6DDSNHPp9CNphSpgqPKXwUrq0ztsyCMXCOB9KvhbFVmSx1yEjHkxk/uLozZcmyWZQFz2lh0OVmqv4h/V2XtTC4p7f6v2Ce6/8RSQ89bYDEdnUBgbWHKEKKBDneA39Ojk5XWZf9GFLvlry9QzI1ztwHMusRB1Zd5Fdql5Jsc4WQhlOczIYjpIbR73TTXhgUvXhvg8NskJ+5RvhuKlharwj5p8hz5Ql3NV1ZIxalpDuOZdO54qsSYK2I8dApIPCjwsdYzJAEgLpok/J18ijy16E3H6K3fIZ4cenF9DPSK/QX4zziiRQEKz6GwRzhl/z1gtG2/tXKVEjo25Md5UdhY+ENLj6eaKDpmBfh2au7JCs4soSNdKJw+YbPSSGOzHY7oG2LyhS0riU4yapW+mckeMHgbbgI6gj4HB1z7hZzL0Futw2HpaXlqn83AzJinSCBAr/zk4YLj3bNxNN/8rJfEjWFMkEp9dQ1fNbwibWbsZu0glEIu3Vis7NLKGwwGpmzhr9RFTeD9w6qJzHFAQd+y/s7hEMc26MdEavK6zhpvCoh8K16hftbtWyrlSvg+Y9M3IyH4n0Dz4Jp5VbTnkhOLyDQjY2oWsmNvOgfHmvcd6TE7nKyY1H2PUTkqZmE6MvTVAm6VlARS6yMoKVWn26GQa9WrnT0hWrDb+GzWryUHUpWo48zQ05xUtemluWSJg2MMtvrGIOBmWuhTskSYlpAJgX45I15WGzhJIKR58kFYBGQw8MrTg1Pz2DrAoKaWPqWOpp+ixzxconr0haeKqxuswOmyj9npfEs9v+j9TFTkc4coQgocg1+bvjhn9EWB6z26eeatwSUnCWV0At/a9CraJflVPRDYeCh7L+z+qpsUi7ztM+9tY/0UCKgAii6Aj1daVH4AExpzHEaXQ+rxTjiOLp3GbtxKCdGLQTg087MZjqItF9oj2WXI74nykUk4PZX5Br1zCPsL4wwWIdcaCJnRPyNfEBYc9YT8BK4ySjH1yAV491vq5RrcxFWAgAoPCrsZ5rCB1d8kwNiQL65kSBliP/x3HkO6YPfLuaLH9zM1kmKWbjlPxO2K2oAc0b4CzxgOsqDfRu6QBAPGy4zxy7FlRFO2ZCrFn+HQV2+SuIu5unhf7oLc8T4c4iyxeO7+hrR1oWjVK/eTUfm/QBu8UkF8JTz2RJ7TUTvfNi88tVER85luj1AFsUjQSMOy80e6mxJER03gXl1a56lHhEjJX+AR2mDSyS/0bKgHvyGQMUmEpwhlp8fpQMDgyRnVQY0ILBNbj5s2Kt63znIyxZe4l62GwVRJoUwqZGbkYDPbKvtwrQ1YS2ZaotU22Z6idkqvLiWfUuB7lwTDak8LmKoh2nwGne3q62m7BQ47ch44MJLqN4yhvQ2JQekI9Axp14q5Z1tqzzqaUWgPWMIdXULa+58S4rgdzUDBVruXDhD8gaEFuuDM/dn6sJTZNDNkWWrjJI93RzAqI97qEhDjYhMTPmFwX0UH71yT3Lg7nTKREkEkNKo0H0Zj6A/pMR0heyKMwz+etY0f7LN71lWoRPW17U8qJnxouiM/J9U1wZnP6kj9+9JvIA2yid7G3jw9fdH0l/4VOwpX8eKTKO3K5PpzxfbGxkZZMZYiJBA1jYsMqH2H0bQbV6PM7NuM4oH5MKec6SYQfHn72zRdiwwmhBmMLP6BWl5E87ldgLkuPJ2+PzDT6t3/3x1camFAhoDib0it9FDn9U8KTbqFdnJ29P1cufwoMTjfhuBmUBSDoK9HYE7LFzrXPipD3kui93C2dzy4Sgu2G2x9ud0BPp5n9vqv9R31IfD1MW7dy/jZORcoqSHS/OvEwriYVOhqxJ3yI/wkO5nhEeiiYXXrSrbqU7ZHYFxaXzSFl5Q5GFviBTY0+uNm6rud7bNx3orCPCuVlzWGcoYaNz6djt9CjHNEInnNDwPkyBFDsoJS5s3KTrehT1purmE95OF80G8cV6DDrciRQYHxEXSNyow+gnXmUrrazSTIW/5R9y4m2IYKsEHf2Ed/Sam7wLyI6c3qfZgTc5XDmvwNjpTA+KXT90LlrCveM5krdkg1xurwJqDfmFkj4bRBMvj3qXO1dJNuagVw9uNgFh5iE+5915WZpyUB6JZCtB2igWYBk1/p+VOJ/CqfyeeAZeeG4CreFYGgGACR+08VDSfV8HjV31x/Ckj70DH59/wnSFZoiYhA+yGU3NV8iyBIkTTrAyXrZMvmXyLZN/BCbfaChFfIvcMPbbE+ysh51vPoXffE3lOHhvHrmnLtopl3336plUT/ioPUTom+H34uMLetDpC5wcbWntc6O1n+0AvajYylKixytJr8bsMs2kh3lnjclZJdpaFsJb77pNvjxfC2j6x/JMTa3m3N0VWNpgHAmEOPpQ2307JJICi5a3AgIhxmBj2b/wHqupBwvURpzGrddUexIXZbWNV/4iap9iJwRrfZX0sd5aKWAjerQKEYTWgFCaG6SFzZHWgx0DMgEdKiLV6GWZo9fk0A7R9c2nvnX+VnIRs0x89tikJmRXmI8FVkwrCWHLgqgDp1jwcbu1Cw8V74QnW5g6/KFc9A7zyfvvnMv5LiJOdtAgmnhVzrs99VIKZDqtz+dzZmgyl7KT17H8JXLfks+WfLbk8zHIJ1rwv5ZTVUgyc/M0S+dT51v6h5qTPSBUuCs940P18DWkFN7zJSuRaWLoDUFe80Yjn5OaycF6ORqAKi2yVtsSzedHNJ/Apvf6JdZItRUJXLk62ngQiJ75jIkpCbfETrnPpoQOqiXBPL9frSTAOGpPdH2fjfOV9srRSJIdmKgFy33CW+fgNkDpVVbxhWjS6+XDg4vLOTXH+Ueyqgfb+4NJ10syfzLYxgF7mQVQQDL/Ki1FI3ND4cyTFUTppKVJLU16bjRpL7dc5031jEw8fFoaiAnS3OS0tcDXb3RPpAjSGQL/Xn1ak/fC17GlrhqEFzn/ImQxqZk1UpOaT77IiXouZ4EwMtFc2M2fpUX9aBg7wZOpRPjaTLOcVDBOsr6OCvmwMsi1GArZqqtyrgwUeus37kXjVXGdYkKeYsh19+hbnEMg+OEjSbtEORnRja06B67W4bgX0ga5WlDQLz1NViPVefhbCi9bQ/LJUDS894A1OSwTIViYu3FOIb+1om9kp5lPO8Q2YpZcetNflKlXXkfKEszTwSQnx/dvn7mQjDP6wWTaPMiAlxk1xSn9IkP2J8ZJQtxVb0q2+4OPg6SESauNqsaQ76FNl/qw2VGnSIdkPxIkCm644lmCUfpnmv0m2eBCZThW4DYFYy9htSTfd9oyAFG4yb+TgiifjCY7k4OxFOA2b04wXQI648+ZMRdYCggUUl4hokZCAMdO2A51oZiNR0IET1eixIwLkBSENuHYNJ4GHKFYgg9Fb28YVRdqOi+gcqpMZ/RuCh0E+fl6jHE0QY6np/GFnfrDpMtI63qnVBVr995oHdZWgSlHFM25zJJbhlENkJW6fsIJCnsrfBjDxpfggadc47SildOz9a7Iu42gXhht3rdFDjoU4F/7C1Fj1PL5kvR5qbmp0Dsr5BNlJIDzkfk71Q1+KV7+1vnwirlRkU0ZwZLWWs8kz4mPbVFvt3pGCoAhqRQZ1l18hon2BrGKzeecdsD+DRnIi7DDcJsB7UncBXBkXchvET8wkj2OEFEYjEuDCbh88FrvZ/PwmMbczoHjyXW+Gt+JBeNvd7bJx8lxwBwxNxNQ9sulawYWVdVZAF2nWrJqqLu+5iNyRO1kqp1MtZOpx5lMLdEQX3fgKhMX96vG2VzvtcKESJWzYaMQgFrgIeSXlNZynQA10Ue3An83F2pi66YustDahc5x44EVz5BoXi2d8BIwWdlYJ5r1UmLt+pJLGvzqX7GtDh21w9Qfv7vA1OkjGvQttJS+pfR3Ts74672TM1Zxe4HOgiHJ4qrgf7ie1k/0jBTqAbAMulqBoS1XLSOgl+Ws4HeS6cRzhyMyF65Hykj3+rxr+sd3EqfvqthrpwW3K2TgW7jys3pfI9rXY3h2VH0hHWKWczUBnYTzoFPRlCvNCsZOPgQL8nbw+NT9QTke+z5pY2A9JaI/hCRvG8GDVJR+Yy0+Nh98i8If7bYeTLJfZVxVIsBiqZTXwhxuKcNjFBtqLfI7GOlmy7tb3t3y7k/Eu30RhqnJxxCh7HLTsHJkGrAi3pEgsY05yLWbGP/uhUBgPUk4677B8T1kRaImoitClZ2907fyRUuJW0r8JNJI9m6xIslRgTG2JfLnAkFezYPl7an6RXfmlXtRsyMSs+YLoJLbp4lIBSP2y4sjk4JFPIMVu2ZTXjlguTCbZ7QjgIP6Y80TziU5zwrJ3h3J9p70n4cnY5ULSiZhLb5fOvFDz2ep9tGuY+AitShmRY6i5nHXAfo65rV3vMy89o5b5tUyr+fCvMLyhi4kGpviKst5acIfh0U5F1/ZbVIUM7e7s3N1dbU9zrJxYrYp7gOkGYG3jzwvojplWthELh8wfRLvGNJmS1vHx76UbLLQwEznzhc+ent2JOmCEzO4cFhk1MnOSU/tH/cI8PTptiJwHo78ZKvDX0QXlPvcX3WWZaSW86N9ieHdsphgq5KEiJshUaFdro4Zm0K94pGRFR2e4iqZnG9q+LrZRa7jzWs9Wp3vnaqvt0idJEtiOROsgG70fiJM03+2uns/ENL3ftgE35lleaG++ebrupK7vHtbfQMqc37Ui9ogR0M8yYJuKoJXPueVJBL4tATMeXy4jmqAGlbIfsUB9hJV2IvwNa/LzKdTA79C73IcHi7MnPQ8zjACJlHfNgfnmKzx2Zfw1tfn56fq1cE5cyXq2Lb6y3YtKBmEqp04RuaflABrijLnKCQtffXFF+rkByhqhno5Io7X52+OSPfzJNPDbfXXlZjgZ3CHfCiPxK/aP3mjCopzHTUyZGH03F6vt/N9D/NwmwxDf4BB/xzXPsNVX/IdQHozJwecw6gSOzKD+YCiGRiKTcjNEYoB11PqKYzlT+p177wnyGWUAppnpt6990MDnBnK8ihAPMpwrZEc1CfvR1wbMGVYQ840/qzP6dmHp4JMRuA7QuDrgJ5dtYTAGlBcf0rPqalvIMnUx+4IoAxIwDFqMMYZOyxffBVwMh8lKbXTfEmNvRpKHoWMugpWZyLWXWBPSqRFv0A/0HxqEsZdT+B2KnDzsg6wOvOY2mWs7AgQcGV96qa2oOYYWi89rM4YFBwWAKI/ATgeTK4GGXSS6HkoqCoFWgRCYyM4onafbrrLglNEXIE8uJ70MOOagPTZ4WmcgbLCqUX4iJ9sOC1kBizpugYFaGGcHbLgVkTz1GJclkK0XSm46TS8v4ieD6olfe6INmdikUGHfu4eNP9JsmLQ252vmAfgb19X5Pcjz2QxoH++PdxTG2/3TzfjCiP7x8oMx4Y9QlhZz3GG0XGhwY8ozkEzYJqoa5qUrOj9U19r3xMaB2/YgeNCVKa/QUyimODxKwVh8UfO4zArz8JRBCczswcnwTf9Mqe2L/tT/jj2h1zkYsF98XUSt3FOeHDJBT3KFKfKAek0peVTPKIjrtrnJ5G5DreKbMuw7I2+GGZXxBwGeeb4fvCYEvJpnNksCfwiOtvZTpDaCVI7QfpEE6TgK4Rxc5ULcTMXGdeG5WwLzgaRGRAFC5q8DCdZlg4n4YuR9VE0BNv+xPwaldTwjVd9wtdcJ0cSSbKwbjYzfBOQzwG55WUULXduufMn4c6f7dqLZRI9Ig/WZM09njkHnNMjYqbLbDZMiCtjK/gU4IwkddckkDsx9Z1e70gxNpDYMJsQuprcHVjorWDTNRub2ZlJbGrURsWwd1iNm/dY8EfnAYbGJML9saT2YZdS6BmuShhWNTUQLEbkkeSW7/okp7uwMxySXCkBWAyFgwdXTb6Vb1uitrdzVw+mqyfiUyQ35Qr1z8lm7BTAJdOPA63QzF1+chXcQ/YyRuHg81H5u/q6pactPW3p6aekp/FCLVeOQNn1OcDOC/bKJAQBsgyY5YzIWY5qEykHSITBkAnwzm59Z+W6GUKp9bhBFMUxmsQW7RptyzP/KJ75BJI1gNxUX9pQtyW34zH2rZrMlFfV6ujXaTKLTmMKuBMWU9csvd2D0R0i4pEXggMuskGWUGgkp0nOHbzoHokcsGjuWHU96hp38gzuyZAYV61g6r4rCWIPv37szp4PnG/J3eHD1Z7qwVzvOLoWQwQjqvy3ybMo8LPLCvJYV9ys111R26zXbUlbS9qeC2kD8HjfOSrnzpOYXFOXs9EoqnrmqrJnWr3kbR+C0Eub4ih5z8C5qnNkMVZ5nGdmuPUyoe7z55uyaE+2TBI5x7mXdSyufsIFRaiTjS+pAcIAr1HgNkCZwn6E78V6gSW7zkn4QwP/JmHEca8kCZ5GOUd+LpePRBDmI2XpPKpvKFeTpfSudBPTZVdsDVA1bWhI50JgJFNL6SE2wnB8J0GVMGiYaCEDflt9V+YoPDElgHbktb6WmRkRIbJSBMSVM6ZPOYhQlR1C4yDd70z1RyKs2UU5o+ZqUffOQ6WSpmSdXPtxJExkU41LTc0WpArvUU42yNGr5qBYhhAcVm2lWqRPbM58ll4lJZKoHP23OFudg2X5azrIJvpcOKAamh+QoyE7Lng+NOBdWb5DqJC/hdGKjrglZCgX/uXXU+ubsCE3RQoYmIMwHqqjxTEiRK284sUaF9VOjL6cLypVbZRpAp8ICXMZfEHzIp5i6VQl7LxtLaual+2CviNwbG5fZ1/+TpUFDHQYAGJnQS7XaL+CiIuM5xp5yZ0rYZys8WVQ2HRLRk7eg+0juQYhFQhGhPR8Z2CkCL6HfYQNvuSC87HdE85sYFwW4TY/iN81fZYMbEckzfiD7ly4yqKzoiEPUBeQuYBJTkcTiBEOG1KN100j9xGr/Rpw9BEs17YRQODW+oVa6U3Iyz2PqzyC1/onyYB4448RkB7Itoe7EqRSpAOzHjFx2UjMiGJHjssKO/7GaJQySLJ8E9dyRErRea7h0WcatD8S0vf6UnPbb1AppLLO6vf08f+TG0sauL50NziYp1BH5HCIXES+vmQZkiECVV4L3IXLpuK2DiymVGzoodOjW0UvLvFxTWDhnI1r4gO+v1MIwA/u6eAf+2aUuF4IU35O9qihtkhgxFWtUWOlNtdO49ppXDuN+yOmcRI02cc1rk3pywvBp2vvVFcSOXRy00rdxESTMU40Oo9jjFVjE1yUYgsvyqlppw/t9OHxpw+fLbkjMhrbnEHEfFy0KtIVG7I1Dbtr3sa6GUKk7xoDOECnXWyYKEPvbSHuYmOm0AC2xn7V0uygE6Dcuc08oXrPsbZciD2xZiTVXllmUy4lX/cScygroJLFF3pnlt9jV+FsLX1E4dZ7VBGfNBS+qpfVKo2juKTpE/fUb3U5M3xP/LBibi/teOtE3ObEJJIj0ZCOqC7WF3Mpia9BEA9l3zeR5usWvO7jnB5Mjms5uvl0VmQ4kx5EKg35Ch9ROg8IFszE1MhcWNdb6cc7KoUlIR3buXK6jEyvCwm/LZ9u+XTLpz8zn14d9fk6MUSIce6jquGpr+GC1kqPyXjKBFvevkxJ06ppIMzH5BO0Qx+kN92m3dLolkY/09yVlxk5Yr5JsC7rwdxNSfoOGQpL0+fuoPiOXb5F+1Zccb/Z6q3J4RHvhXHG7hCJ2JKhkMb5CV7OTzfRJIzhV2Ii1UnJT0ngVtojsbpNPh+37K8eTM6+S+xshuRBHuI1uSRTMrEiYkz/B+JCzLRoxQAA"

EXPECTED_HASHES = {
    "train": "4696a7ca5de38e4dac9d08ab881ea3627db2a673ec000118cf95d875224f637a",
    "val": "8c0cfce71f003c0d7ec8a2a216b8f05d38af913c681f5ac45971b9c0c93cab79",
    "test": "aa8ece32b92324191f0536934129fb6be41d2a4774c992f65a9dc5b73fcb86d2"
}

for split, b64_str in [("train", TRAIN_B64), ("val", VAL_B64), ("test", TEST_B64)]:
    target_file = f"dataset/{split}.jsonl"
    raw_bytes = gzip.decompress(base64.b64decode(b64_str))
    calc_hash = hashlib.sha256(raw_bytes).hexdigest()
    assert calc_hash == EXPECTED_HASHES[split], f"Checksum mismatch for {split} split!"
    with open(target_file, "wb") as f:
        f.write(raw_bytes)

print("=" * 60)
print("       CRACKPROOF DATASET INTEGRITY VERIFIED")
print("=" * 60)
for split, expected_count in [("train", 108), ("val", 18), ("test", 18)]:
    with open(f"dataset/{split}.jsonl") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    print(f"  ✔ dataset/{split}.jsonl : {len(rows)} samples (Expected: {expected_count})")
    assert len(rows) == expected_count, f"Unexpected count in {split}: {len(rows)}"

print("\n[OK] Pydantic AnswerEvaluation schema loaded and verified.")
print("[OK] Dataset ready for QLoRA fine-tuning.")


## Step 3: Load 4-Bit Quantized Base Model
We load `Qwen/Qwen2.5-3B-Instruct` using `BitsAndBytesConfig` (4-bit NF4). This fits comfortably in ~3.5 GB VRAM on the Colab T4 GPU.

In [ ]:
import random
import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Set seeds for reproducible model and adapter initialization
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
transformers.set_seed(42)

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID} in 4-bit NF4 Quantization...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

print(f'[OK] Model successfully loaded on GPU! Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')


## Step 4: Attach LoRA Adapter (PEFT)
We freeze base parameters and attach LoRA adapter matrices ($r=16, \alpha=32$) on attention and MLP projections.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 5: Execute QLoRA Training with Completion-Only Loss Masking
Fine-tunes the model on our ChatML interview evaluation dataset using 4-bit QLoRA.
We use **completion-only loss masking** (system, user, question, candidate answer, and RAG reference tokens are masked with `-100`; only assistant JSON tokens are supervised) and dynamic padding via `DataCollatorForSeq2Seq`.


In [ ]:
# =======================================================
# STEP 5: Execute QLoRA Training with Completion-Only Loss Masking
# =======================================================
import random
import numpy as np
import torch
import transformers
from datasets import load_dataset
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

# 0. Set explicit seeds for deterministic reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
transformers.set_seed(42)

# 1. Load the Question-Level Split Datasets
dataset = load_dataset("json", data_files={
    "train": "dataset/train.jsonl",
    "val": "dataset/val.jsonl"
})

# 2. Completion-Only Loss Masking Tokenization
# Masks prompt tokens (system, user, question, candidate answer, RAG evidence) with -100
# Only supervises the assistant JSON completion tokens
def tokenize_completion_only(batch):
    input_ids_list = []
    labels_list = []
    attention_mask_list = []
    max_seq_len = 1536

    for messages in batch["messages"]:
        prompt_messages = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

        # Prompt tokens -> -100 (ignored in loss computation)
        # Assistant tokens -> supervised token IDs
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

        if len(full_ids) > max_seq_len:
            full_ids = full_ids[:max_seq_len]
            labels = labels[:max_seq_len]

        input_ids_list.append(full_ids)
        labels_list.append(labels)
        attention_mask_list.append([1] * len(full_ids))

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list,
    }

train_data = dataset["train"].map(
    tokenize_completion_only,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
val_data = dataset["val"].map(
    tokenize_completion_only,
    batched=True,
    remove_columns=dataset["val"].column_names,
)

# 3. Dynamic Padding Collator with Label Padding (-100)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

# 4. Deterministic Training Configuration
training_args = TrainingArguments(
    output_dir="./crackproof_qlora_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=2,
    save_strategy="steps",
    save_steps=2,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    data_seed=42,
)

import inspect
trainer_kwargs = {
    "model": model,
    "train_dataset": train_data,
    "eval_dataset": val_data,
    "data_collator": data_collator,
    "args": training_args,
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Starting QLoRA Fine-Tuning on T4 GPU...")
print(f"Training samples: {len(train_data)} | Validation samples: {len(val_data)}")
print("Effective Batch Size: 2 * 4 = 8")
print("Evaluation Strategy: eval_strategy='steps', eval_steps=2\n")

# 5. Train the model
train_result = trainer.train()

# 6. Save best adapter and tokenizer
trainer.model.save_pretrained("./crackproof_qlora_adapter")
tokenizer.save_pretrained("./crackproof_qlora_adapter")
print("\n[SUCCESS] Fine-tuning complete! Best adapter saved to ./crackproof_qlora_adapter")



## Step 5.5: Plot Training vs Validation Loss Curve & Identify Sweet Spot
Extracts  and  from .
Calculates the **true sweet spot** ($\min(\text{eval\_loss})$) and checks for overfitting.

In [ ]:
# =======================================================
# STEP 5.5: Plot Training vs Validation Loss Curve
# =======================================================
import matplotlib.pyplot as plt
import numpy as np

# 1. Parse log history
train_steps, train_losses = [], []
eval_steps, eval_losses = [], []

for entry in trainer.state.log_history:
    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

print(f"Logged {len(train_losses)} training loss points and {len(eval_losses)} validation loss points.")

# 2. Identify Best Observed Validation-Loss Checkpoint
best_val_step = None
best_val_loss = None
if eval_losses:
    best_idx = int(np.argmin(eval_losses))
    best_val_step = eval_steps[best_idx]
    best_val_loss = eval_losses[best_idx]
    print(f"\n🎯 Best Observed Validation-Loss Checkpoint: Step {best_val_step} with Eval Loss = {best_val_loss:.4f}")

# 3. Check for Potential Overfitting Signal
if len(eval_losses) > 1 and best_val_step is not None:
    if eval_losses[-1] > best_val_loss:
        delta = eval_losses[-1] - best_val_loss
        print(f"⚠️ Potential Overfitting Signal: Final validation loss is {delta:.4f} higher than minimum at step {best_val_step}.")
    else:
        print("✅ No Overfitting Signal: Validation loss improved monotonically through training.")

# 4. High-Resolution Publication-Quality Plot
plt.figure(figsize=(10, 5), dpi=150)
plt.plot(train_steps, train_losses, label="Training Loss", color="#1f77b4", linewidth=2.5, marker="o", markersize=6)

if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Validation Loss", color="#d62728", linewidth=2.5, linestyle="--", marker="s", markersize=6)
    
    # Highlight the Best Observed Validation-Loss Checkpoint
    plt.scatter([best_val_step], [best_val_loss], color="#ff7f0e", s=200, zorder=5, edgecolors="black", linewidth=1.5)
    plt.annotate(
        f"Best Val Checkpoint: {best_val_loss:.4f}\nStep {best_val_step}",
        xy=(best_val_step, best_val_loss),
        xytext=(best_val_step + 0.2, best_val_loss + 0.04),
        arrowprops=dict(facecolor="black", shrink=0.08, width=1.5, headwidth=8),
        fontsize=10,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff2cc", edgecolor="#ff7f0e", alpha=0.9)
    )

plt.title("CrackProof QLoRA: Training vs Validation Loss Curve", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Optimizer Steps", fontsize=12, labelpad=8)
plt.ylabel("Cross-Entropy Loss", fontsize=12, labelpad=8)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=12, loc="upper right")
plt.tight_layout()
plt.savefig("crackproof_loss_curve.png", dpi=300)
plt.show()
print("[SAVED] Loss curve saved to crackproof_loss_curve.png")


## Step 6: Benchmark Test (Base Model vs Fine-Tuned Model)
We test an authentic candidate answer to verify that the model produces structured JSON with the 4 SOLO dimensions and RAG citations.

In [ ]:
# =======================================================
# STEP 6: Full Benchmark on Frozen Test Set (18 Samples)
# =======================================================
import json
import time
import torch

with open("dataset/test.jsonl") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(test_rows)} frozen test samples across 6 subjects (Java, OOP, DBMS, OS, CN, DSA):\n")

model.eval()

predictions = []
correct_json_count = 0
valid_schema_count = 0
verdict_matches = 0
correctness_errors = []
depth_errors = []
valid_citations_count = 0
total_citations_count = 0

for i, sample in enumerate(test_rows):
    user_msg = next(m["content"] for m in sample["messages"] if m["role"] == "user")
    system_msg = next(m["content"] for m in sample["messages"] if m["role"] == "system")
    target_eval_str = next(m["content"] for m in sample["messages"] if m["role"] == "assistant")
    target_eval = json.loads(target_eval_str)

    test_messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    formatted_input = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)

    start_t = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = round(time.time() - start_t, 2)

    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    clean_text = generated_text
    if "```json" in clean_text:
        clean_text = clean_text.split("```json")[1].split("```")[0].strip()
    elif "```" in clean_text:
        clean_text = clean_text.split("```")[1].split("```")[0].strip()

    is_valid_json = False
    is_schema_valid = False
    verdict_match = False
    parsed_dict = None
    err_msg = None

    try:
        parsed_dict = json.loads(clean_text)
        is_valid_json = True
        correct_json_count += 1
    except Exception as e:
        err_msg = f"JSON Parse Error: {e}"

    if is_valid_json and parsed_dict:
        try:
            parsed_obj = AnswerEvaluation.model_validate(parsed_dict)
            is_schema_valid = True
            valid_schema_count += 1

            if parsed_obj.verdict == target_eval.get("verdict"):
                verdict_match = True
                verdict_matches += 1

            c_err = abs(parsed_obj.correctness_score - target_eval.get("correctness_score", 0))
            d_err = abs(parsed_obj.depth_score - target_eval.get("depth_score", 0))
            correctness_errors.append(c_err)
            depth_errors.append(d_err)

            for c in parsed_obj.citations:
                total_citations_count += 1
                if f"[{c.source_number}]" in user_msg or f"[{c.source_number}]" in sample.get("rag_context", ""):
                    valid_citations_count += 1
        except Exception as e:
            err_msg = f"Schema Validation Error: {e}"

    pred_record = {
        "sample_index": i + 1,
        "question_id": sample.get("question_id"),
        "subject": sample.get("subject"),
        "latency_seconds": latency,
        "ground_truth": target_eval,
        "raw_response": generated_text,
        "parsed_prediction": parsed_dict,
        "is_valid_json": is_valid_json,
        "is_schema_valid": is_schema_valid,
        "schema_error": err_msg,
        "verdict_match": verdict_match,
        "predicted_verdict": parsed_dict.get("verdict") if parsed_dict else None,
        "true_verdict": target_eval.get("verdict"),
        "predicted_correctness_score": parsed_dict.get("correctness_score") if parsed_dict else None,
        "true_correctness_score": target_eval.get("correctness_score"),
        "predicted_depth_score": parsed_dict.get("depth_score") if parsed_dict else None,
        "true_depth_score": target_eval.get("depth_score"),
    }
    predictions.append(pred_record)

    status_str = "✔ PASS" if (is_schema_valid and verdict_match) else "⚠ DIFF"
    print(f"[{i+1}/18] {status_str} | {sample.get('question_id')} ({sample.get('subject')}): Pred Verdict={pred_record['predicted_verdict']} (True={pred_record['true_verdict']}) | Pred Score={pred_record['predicted_correctness_score']}/10 (True={pred_record['true_correctness_score']}/10) | Latency: {latency}s")

n_test = len(test_rows)
json_rate = round(correct_json_count / n_test * 100, 1)
schema_rate = round(valid_schema_count / n_test * 100, 1)
verdict_acc = round(verdict_matches / n_test * 100, 1)
c_mae = round(sum(correctness_errors) / len(correctness_errors), 2) if correctness_errors else None
d_mae = round(sum(depth_errors) / len(depth_errors), 2) if depth_errors else None
cit_prec = round(valid_citations_count / total_citations_count * 100, 1) if total_citations_count > 0 else 0.0

benchmark_output = {
    "benchmark_metadata": {
        "model": "Qwen/Qwen2.5-3B-Instruct (Fine-Tuned QLoRA)",
        "test_file": "dataset/test.jsonl",
        "total_samples": n_test,
        "metrics": {
            "json_parse_rate_percent": json_rate,
            "schema_adherence_percent": schema_rate,
            "verdict_accuracy_percent": verdict_acc,
            "correctness_score_mae": c_mae,
            "depth_score_mae": d_mae,
            "citation_precision_percent": cit_prec,
        }
    },
    "predictions": predictions
}

with open("benchmark_finetuned_predictions.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_output, f, indent=2)

print("\n" + "=" * 65)
print("       CRACKPROOF FINE-TUNED BENCHMARK RESULTS (18 SAMPLES)     ")
print("=" * 65)
print(f"  Valid JSON Output Rate : {json_rate}% ({correct_json_count}/{n_test})")
print(f"  Pydantic Schema Match  : {schema_rate}% ({valid_schema_count}/{n_test})")
print(f"  Verdict Accuracy       : {verdict_acc}% ({verdict_matches}/{n_test})")
print(f"  Correctness Score MAE  : {c_mae} points")
print(f"  Depth Score MAE        : {d_mae} points")
print(f"  Citation Precision     : {cit_prec}% ({valid_citations_count}/{total_citations_count})")
print("=" * 65)
print("[SAVED] Saved full fine-tuned benchmark to benchmark_finetuned_predictions.json")


## Step 7: Package & Download Trained Artifacts
Zips the `./crackproof_qlora_adapter` directory, `crackproof_loss_curve.png`, and `benchmark_finetuned_predictions.json` into a single zip file and triggers a direct browser download.

In [ ]:
# =======================================================
# STEP 7: Package & Download Trained Artifacts
# =======================================================
import os
import shutil
import zipfile

# 1. Package adapter directory
os.makedirs("crackproof_qlora_adapter", exist_ok=True)
shutil.make_archive("crackproof_artifacts", "zip", root_dir=".", base_dir="crackproof_qlora_adapter")

# 2. Add extra benchmark artifacts to the zip
with zipfile.ZipFile("crackproof_artifacts.zip", "a") as z:
    if os.path.exists("crackproof_loss_curve.png"):
        z.write("crackproof_loss_curve.png")
    if os.path.exists("benchmark_finetuned_predictions.json"):
        z.write("benchmark_finetuned_predictions.json")

print("[OK] Created crackproof_artifacts.zip successfully!")

try:
    from google.colab import files
    files.download("crackproof_artifacts.zip")
    print("[DOWNLOAD] Downloading crackproof_artifacts.zip to your computer...")
except Exception as e:
    print(f"Direct download note: {e}")
    print("You can manually download crackproof_artifacts.zip from the Colab Files panel (folder icon on left).")
